# Notebook to Compare Heuristics

In [82]:
import random
import networkx as nx

from time import time

random.seed(42)  # For accurate comparison

In [83]:
from src import (
    kempe_greedy,
    welfare_greedy,
)

from src import (
    estimate_influence,
    independent_cascade_community
)

## Comparison of Kempe Greedy and Welfare Greedy

In [84]:
graph = nx.erdos_renyi_graph(
    n=200,
    p=0.05,
    seed=42,
    directed=True,
)

# Assign communities randomly
for i, node in enumerate(graph.nodes()):
    graph.nodes[node]['community'] = random.randint(0, 2)

communities = set(nx.get_node_attributes(graph, 'community').values())

In [85]:
k = 5  # number of seeds to select
alpha = 1.5  # inequality-aversion parameter (higher = more fairness)
p = 0.1  # edge activation probability
num_sims = 1000

In [86]:
start = time()
kempe_seeds = kempe_greedy(
    graph=graph,
    k=k,
    probability=p,
    num_simulations=num_sims,
)
kempe_time = time() - start

kempe_influence = estimate_influence(
    graph=graph,
    seeds=kempe_seeds,
    propagation_prob=p,
    num_simulations=num_sims,
)

kempe_by_comm = independent_cascade_community(
    graph=graph,
    seeds=kempe_seeds,
    probability=p,
    num_sims=500,
)

kempe_by_comm = {k: round(v, 2) for k, v in kempe_by_comm.items()}

Selecting seeds: 100%|██████████| 5/5 [00:20<00:00,  4.17s/it]


In [87]:
start = time()
welfare_seeds = welfare_greedy(
    graph=graph,
    communities=communities,
    k=k,
    alpha=alpha,
    probability=p,
    num_sims=num_sims,
)
welfare_time = time() - start

welfare_influence = estimate_influence(
    graph=graph,
    seeds=welfare_seeds,
    propagation_prob=p,
    num_simulations=num_sims,
)

welfare_by_comm = independent_cascade_community(
    graph=graph,
    seeds=welfare_seeds,
    probability=p,
    num_sims=500,
)

welfare_by_comm = {k: round(v, 2) for k, v in welfare_by_comm.items()}

Selecting seeds: 100%|██████████| 5/5 [00:21<00:00,  4.32s/it]


In [88]:
print(f'Kempe Greedy:\n Seeds: {kempe_seeds}\n Time: {kempe_time:.2f}s\n Average Influence: {kempe_influence:.1f}\n')
print(f'Welfare Greedy:\n Seeds: {welfare_seeds}\n Time: {welfare_time:.2f}s\n Average Influence: {welfare_influence:.1f}\n')

Kempe Greedy:
 Seeds: {65, 106, 113, 54, 154}
 Time: 20.87s
 Average Influence: 42.4

Welfare Greedy:
 Seeds: {97, 71, 115, 184, 154}
 Time: 21.59s
 Average Influence: 37.5



In [89]:
print(f'Average Influence by Community (Kempe): {kempe_by_comm}')
print(f'Average Influence by Community (Welfare): {welfare_by_comm}')

Average Influence by Community (Kempe): {0: 0.38, 1: 0.31, 2: 0.31}
Average Influence by Community (Welfare): {0: 0.32, 1: 0.33, 2: 0.34}
